# Time Series Forecasting — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/time-series/notebooks/time-series-lab.ipynb)

Companion notebook for the **Time Series Forecasting** track (`ts-m1` … `ts-m6`).
One series runs through the whole notebook, and it is **the same series the
interactive widgets on the site draw** — the generator below is a line-for-line
port of the site's, PRNG included, so every number you see here is the number
in the module text.

Two implementations of everything: **by hand in NumPy** so the mechanism is
visible, then the **statsmodels** call you would actually ship. Where the two
disagree it is because they use different estimators, and the notebook says which.

CPU only, no downloads, runs in well under a minute.

| Section | Module | What runs |
|---|---|---|
| 1 | ts-m1 | components, additive vs multiplicative, decomposition |
| 2 | ts-m2 | naive / mean / moving-average baselines, RMSE, split discipline |
| 3 | ts-m3 | SES, Holt, Holt-Winters, fitting α/β/γ |
| 4 | ts-m4 | stationarity, ADF, KPSS, differencing, ACF |
| 5 | ts-m5 | ACF/PACF identification, Yule-Walker, widening intervals |
| 6 | ts-m6 | ARMA, ARIMA, SARIMA, Ljung-Box, the Box-Jenkins loop |

## Setup and the series

`mulberry32` is the site's seeded PRNG. Porting it (rather than using
`np.random`) is what makes this notebook and the widgets agree to six decimals.

In [ ]:
import numpy as np, pandas as pd, math
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams['figure.figsize'] = (11, 3.2)
TRACK = '#0369a1'

def make_rng(seed=7):
    """mulberry32 — the site's PRNG, ported exactly (32-bit wraparound and all)."""
    s = seed & 0xFFFFFFFF
    def rand():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = s
        t = ((t ^ (t >> 15)) * (t | 1)) & 0xFFFFFFFF
        t ^= (t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        t &= 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    return rand

def randn(rand):
    """Box-Muller on top of the uniform generator."""
    u = v = 0.0
    while u == 0: u = rand()
    while v == 0: v = rand()
    return math.sqrt(-2.0*math.log(u)) * math.cos(2.0*math.pi*v)

def generate_series(n=48, period=12, slope=0.6, amp=8, noise=2, level=40,
                    mode='additive', seed=7):
    rand = make_rng(seed)
    y, trend = [], []
    for i in range(n):
        tr = level + slope*i
        s = math.sin(2*math.pi*i/period)
        eps = randn(rand)*noise
        trend.append(tr)
        y.append(tr + amp*s + eps if mode == 'additive'
                 else tr * (1 + (amp/100)*s) * (1 + eps/100))
    return np.array(y), np.array(trend)

PERIOD = 12
y, true_trend = generate_series()
dates = pd.date_range('2022-01-01', periods=len(y), freq='MS')
s = pd.Series(y, index=dates, name='sales')

print(f'{len(y)} monthly observations, period {PERIOD}')
print('first 3:', y[:3].round(6))
print('last 3 :', y[-3:].round(6))
# These are the exact values the site's widgets plot.
assert abs(y[0] - 45.518746) < 1e-6 and abs(y[-1] - 64.541506) < 1e-6

In [ ]:
fig, ax = plt.subplots()
ax.plot(s.index, s.values, marker='o', ms=3, color=TRACK, label='observed')
ax.plot(s.index, true_trend, '--', color='grey', label='true trend (known: it is synthetic)')
ax.set_title('NovaRetail monthly sales'); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## ts-m1 · Components and decomposition

`observed = trend + seasonality + residual` (additive) or
`trend x seasonality x residual` (multiplicative). Which one you pick is a
statement about whether the seasonal swing **grows with the level**.

**Reading the next cell.** Two details do the work:

- `centered_ma` handles an **even** period (12 months) by averaging two offset
  windows. A 12-point average sits *between* two months; averaging two of them
  shifts it back onto a month. Odd periods need no such trick.
- The seasonal index is the **average detrended value at each position in the
  cycle** — all the Januaries, then all the Februaries. Subtracting the mean of
  those twelve numbers forces them to sum to zero, so the seasonal component
  moves value around within a year without adding any.

In [ ]:
def centered_ma(y, period):
    """Centred moving average. For an EVEN period the plain MA sits between two
    time points, so you average two offset windows to re-centre it."""
    n, half = len(y), period // 2
    out = np.full(n, np.nan)
    for i in range(half, n - half):
        if period % 2 == 0:
            w1 = y[i-half : i-half+period].mean()
            w2 = y[i-half+1 : i-half+1+period].mean()
            out[i] = (w1 + w2) / 2
        else:
            out[i] = y[i-half : i+half+1].mean()
    return out

def classical_decompose(y, period, mode='additive'):
    trend = centered_ma(y, period)
    detr = (y - trend) if mode == 'additive' else (y / trend)
    # average the detrended value by position within the cycle
    phase = np.array([np.nanmean(detr[p::period]) for p in range(period)])
    phase = phase - phase.mean() if mode == 'additive' else phase / phase.mean()
    seasonal = phase[np.arange(len(y)) % period]
    resid = (y - trend - seasonal) if mode == 'additive' else (y / (trend * seasonal))
    return trend, seasonal, resid

trend, seasonal, resid = classical_decompose(y, PERIOD)
print('seasonal indices (one per month):', seasonal[:PERIOD].round(3))
print('they sum to ~0 by construction:', round(seasonal[:PERIOD].sum(), 10))
print(f'trend is NaN for the first and last {PERIOD//2} points — a centred MA has no edges')
assert abs(seasonal[:PERIOD].sum()) < 1e-9

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
for a, (v, name) in zip(ax, [(y, 'observed'), (trend, 'trend'),
                             (seasonal, 'seasonal'), (resid, 'residual')]):
    a.plot(dates, v, color=TRACK, lw=1.4); a.set_ylabel(name, fontsize=9)
ax[3].axhline(0, color='grey', lw=0.8)
plt.tight_layout(); plt.show()

# The same thing from statsmodels, plus STL which handles a changing seasonal shape.
from statsmodels.tsa.seasonal import seasonal_decompose, STL
sm = seasonal_decompose(s, model='additive', period=PERIOD)
print('max |ours - statsmodels| on the trend:',
      np.nanmax(np.abs(trend - sm.trend.values)).round(10))

stl = STL(s, period=PERIOD, robust=True).fit()
print('STL lets the seasonal component EVOLVE; classical forces it constant.')
print('seasonal amplitude, first cycle vs last:',
      round(float(np.ptp(stl.seasonal.values[:PERIOD])), 2), 'vs',
      round(float(np.ptp(stl.seasonal.values[-PERIOD:])), 2))

In [ ]:
# Additive or multiplicative? Look at whether the swing grows with the level.
y_mult, _ = generate_series(mode='multiplicative', amp=25, noise=6)
fig, ax = plt.subplots(1, 2)
ax[0].plot(y, color=TRACK); ax[0].set_title('additive: constant swing', fontsize=10)
ax[1].plot(y_mult, color='#dc2626'); ax[1].set_title('multiplicative: swing grows with level', fontsize=10)
plt.tight_layout(); plt.show()

# A log turns a multiplicative structure into an additive one:
#   log(T*S*R) = log T + log S + log R
print('ratio of last-cycle to first-cycle range:')
print(f'  additive      {np.ptp(y[-PERIOD:]) / np.ptp(y[:PERIOD]):.2f}  (~1 => additive)')
print(f'  multiplicative{np.ptp(y_mult[-PERIOD:]) / np.ptp(y_mult[:PERIOD]):>6.2f}  (>1 => use log or a multiplicative model)')
print(f'  after log     {np.ptp(np.log(y_mult[-PERIOD:])) / np.ptp(np.log(y_mult[:PERIOD])):>6.2f}  (back to ~1)')

---
## ts-m2 · Baselines and RMSE

A model that cannot beat "yesterday's value" is not a model. Baselines exist to
make that judgement cheap, and the split has to stay **chronological**.

**Reading the next cell.** Each baseline is one assumption written as one line:

| baseline | assumes | leading NaNs |
|---|---|---|
| `naive` | tomorrow looks like today | 1 |
| `mean` | the series has no trend, only noise around a level | 1 |
| `moving_avg(w)` | recent history matters, older history does not | w |
| `seasonal_naive(m)` | this month looks like the same month last cycle | m |

The NaNs are not padding — they mark the points where the baseline **has no
opinion yet**. `scores` masks them out, so every model is judged only where it
actually made a prediction. Comparing a model scored on 47 points against one
scored on 36 is a silent way to declare the wrong winner.

In [ ]:
def naive(y):        return np.r_[np.nan, y[:-1]]
def mean_fc(y):      return np.array([np.nan if i == 0 else y[:i].mean() for i in range(len(y))])
def moving_avg(y, w): return np.array([np.nan if i < w else y[i-w:i].mean() for i in range(len(y))])
def seasonal_naive(y, m): return np.array([np.nan if i < m else y[i-m] for i in range(len(y))])

def scores(y_true, y_pred):
    mask = ~np.isnan(y_pred)
    e = y_true[mask] - y_pred[mask]
    return dict(RMSE=np.sqrt((e**2).mean()), MAE=np.abs(e).mean(),
                MAPE=100*np.abs(e/y_true[mask]).mean(), n=mask.sum())

base = {'naive': naive(y), 'mean': mean_fc(y), 'MA(4)': moving_avg(y, 4),
        'MA(12)': moving_avg(y, 12), f'seasonal naive({PERIOD})': seasonal_naive(y, PERIOD)}
print(f'{"model":<22}{"RMSE":>8}{"MAE":>8}{"MAPE%":>8}{"n":>5}')
for k, v in base.items():
    m = scores(y, v)
    print(f'{k:<22}{m["RMSE"]:>8.3f}{m["MAE"]:>8.3f}{m["MAPE"]:>8.2f}{m["n"]:>5}')
assert abs(scores(y, naive(y))['RMSE'] - 4.185604) < 1e-6      # matches the site

In [ ]:
# RMSE punishes a few large errors; MAE does not. Same MAE, different RMSE:
a = np.array([2., 2, 2, 2]); b = np.array([0., 0, 0, 8])
print(f'errors {a}: MAE {np.abs(a).mean():.2f}  RMSE {np.sqrt((a**2).mean()):.2f}')
print(f'errors {b}: MAE {np.abs(b).mean():.2f}  RMSE {np.sqrt((b**2).mean()):.2f}')
print('Pick RMSE when one big miss is much worse than several small ones.')

# MAPE is undefined at zero and asymmetric — it prefers under-forecasting.
print(f'\nMAPE(true=100, pred=50)  = {100*abs(100-50)/100:>6.1f}%')
print(f'MAPE(true=100, pred=150) = {100*abs(100-150)/100:>6.1f}%  <- same absolute error')
print(f'MAPE(true=50,  pred=100) = {100*abs(50-100)/50:>6.1f}%  <- but a 2x over-forecast costs double')

In [ ]:
# The split. Chronological only — a random split lets the model see the future.
split = int(len(y) * 0.8)
train, test = y[:split], y[split:]
print(f'train {len(train)} points (to {dates[split-1]:%Y-%m}), test {len(test)}')

h = len(test)
fc = {'naive': np.repeat(train[-1], h),
      'mean': np.repeat(train.mean(), h),
      'seasonal naive': np.array([train[-PERIOD + (i % PERIOD)] for i in range(h)]),
      'drift': train[-1] + (np.arange(1, h+1) * (train[-1]-train[0]) / (len(train)-1))}
print(f'\n{"model":<16}{"test RMSE":>10}')
for k, v in fc.items():
    print(f'{k:<16}{np.sqrt(((test-v)**2).mean()):>10.3f}')

fig, ax = plt.subplots()
ax.plot(dates[:split], train, color=TRACK, label='train')
ax.plot(dates[split:], test, color='black', lw=2, label='test (actual)')
for k, v in fc.items(): ax.plot(dates[split:], v, '--', lw=1.2, label=k)
ax.legend(fontsize=8, ncol=3); plt.tight_layout(); plt.show()

In [ ]:
# Why a random split lies. Fit on shuffled data and the "test" error collapses,
# because neighbouring points leak across the boundary.
rng = np.random.default_rng(0)
idx = rng.permutation(len(y))
r_tr, r_te = np.sort(idx[:split]), np.sort(idx[split:])

# interpolate from the training points — trivially easy when they surround you
pred_random = np.interp(r_te, r_tr, y[r_tr])
pred_chrono = np.repeat(y[:split][-1], len(test))
print(f'random-split RMSE       {np.sqrt(((y[r_te]-pred_random)**2).mean()):.3f}  <- flattering nonsense')
print(f'chronological-split RMSE {np.sqrt(((test-pred_chrono)**2).mean()):.3f}  <- the honest number')
print('\nThe random split never asks the model to extrapolate, which is the only')
print('thing a forecaster actually has to do.')

# The right tool for model selection on time series:
from sklearn.model_selection import TimeSeriesSplit
for i, (tr_i, te_i) in enumerate(TimeSeriesSplit(n_splits=4).split(y)):
    print(f'fold {i}: train[0:{tr_i[-1]+1}]  test[{te_i[0]}:{te_i[-1]+1}]  <- train always precedes test')

---
## ts-m3 · Exponential smoothing

Every variant is the same idea — a weighted average where the weights decay
geometrically into the past — with a component added per pattern you need.

**Reading the next cell.** All three are the same recursion with more state:

| | state carried | update |
|---|---|---|
| `ses` | level | `L = a*y + (1-a)*L_prev` |
| `holt` | level, trend | level update uses `L_prev + T_prev`, then the trend is smoothed too |
| `holt_winters` | level, trend, seasonal | the seasonal index is **subtracted before** updating the level and re-smoothed after |

The pattern each time is `new = alpha * (what just happened) + (1-alpha) * (what
we expected)`. Note `holt_winters` needs two full cycles before it can start —
one to seed the level, one to estimate the initial trend — which is why
`fitted` is NaN for the first `m` points.

In [ ]:
def ses(y, alpha):
    level = np.empty(len(y)); level[0] = y[0]
    for i in range(1, len(y)):
        level[i] = alpha*y[i] + (1-alpha)*level[i-1]
    fitted = np.r_[np.nan, level[:-1]]
    return fitted, level[-1]

def holt(y, alpha, beta):
    L = np.empty(len(y)); T = np.empty(len(y))
    L[0], T[0] = y[0], y[1]-y[0]
    for i in range(1, len(y)):
        L[i] = alpha*y[i] + (1-alpha)*(L[i-1]+T[i-1])
        T[i] = beta*(L[i]-L[i-1]) + (1-beta)*T[i-1]
    fitted = np.r_[np.nan, (L[:-1]+T[:-1])]
    return fitted, L[-1], T[-1]

def holt_winters(y, alpha, beta, gamma, m):
    n = len(y)
    L = np.full(n, np.nan); T = np.full(n, np.nan); S = np.full(n+m, np.nan)
    L[m-1] = y[:m].mean()
    T[m-1] = (y[m:2*m].mean() - y[:m].mean()) / m
    S[:m] = y[:m] - L[m-1]
    fitted = np.full(n, np.nan)
    for i in range(m, n):
        sm = S[i-m]
        fitted[i] = L[i-1] + T[i-1] + sm
        L[i] = alpha*(y[i]-sm) + (1-alpha)*(L[i-1]+T[i-1])
        T[i] = beta*(L[i]-L[i-1]) + (1-beta)*T[i-1]
        S[i] = gamma*(y[i]-L[i]) + (1-gamma)*sm
    def forecast(h):
        return np.array([L[n-1] + (k+1)*T[n-1] + S[n-m + (k % m)] for k in range(h)])
    return fitted, forecast

for name, f in [('SES(0.3)', ses(y, 0.3)[0]),
                ('Holt(0.3,0.1)', holt(y, 0.3, 0.1)[0]),
                (f'Holt-Winters', holt_winters(y, 0.3, 0.1, 0.3, PERIOD)[0])]:
    print(f'{name:<16} in-sample RMSE {scores(y, f)["RMSE"]:.3f}')
print('\nSES has no trend term, so on a trending series it lags permanently behind.')

In [ ]:
# alpha controls how fast the past is forgotten: weight on lag k is a(1-a)^k.
fig, ax = plt.subplots(1, 2)
for a in (0.1, 0.3, 0.7):
    ax[0].plot(dates, ses(y, a)[0], lw=1.2, label=f'alpha={a}')
    ax[1].plot(range(12), [a*(1-a)**k for k in range(12)], marker='o', ms=3, label=f'alpha={a}')
ax[0].plot(dates, y, color='black', lw=1, alpha=0.5, label='actual')
ax[0].set_title('SES fits', fontsize=10); ax[0].legend(fontsize=8)
ax[1].set_title('weight on the observation k steps back', fontsize=10)
ax[1].set_xlabel('k'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Fit alpha the way software does: minimise in-sample SSE.
grid = np.linspace(0.01, 0.99, 99)
sse = [np.nansum((y - ses(y, a)[0])**2) for a in grid]
best = grid[int(np.argmin(sse))]
print(f'alpha minimising SSE = {best:.2f}  (SSE {min(sse):.1f})')
print('Note it lands near 1: with a strong trend, SES can only keep up by')
print('forgetting almost everything — a symptom that you need Holt, not a better alpha.')

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

fit = ExponentialSmoothing(s, trend='add', seasonal='add',
                           seasonal_periods=PERIOD, initialization_method='estimated').fit()
print('alpha (level)    ', round(fit.params['smoothing_level'], 4))
print('beta  (trend)    ', round(fit.params['smoothing_trend'], 4))
print('gamma (seasonal) ', round(fit.params['smoothing_seasonal'], 4))
print('AIC              ', round(fit.aic, 2))

fc = fit.forecast(12)
fig, ax = plt.subplots()
ax.plot(s.index, s.values, color=TRACK, label='observed')
ax.plot(s.index, fit.fittedvalues, color='#dc2626', lw=1, label='fitted')
ax.plot(fc.index, fc.values, '--', color='#dc2626', lw=2, label='forecast (12)')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## ts-m4 · Stationarity

Stationary means the mean, variance and autocovariance do not depend on *when*
you look. ARMA models assume it; the fix when it fails is differencing.

In [ ]:
def rolling(y, w, fn):
    return np.array([np.nan if i < w-1 else fn(y[i-w+1:i+1]) for i in range(len(y))])

fig, ax = plt.subplots(1, 2)
for a, (v, name) in zip(ax, [(y, 'original'), (np.diff(y), 'first difference')]):
    a.plot(v, color=TRACK, lw=1, alpha=0.6)
    a.plot(rolling(v, 12, np.mean), color='#dc2626', label='rolling mean(12)')
    a.plot(rolling(v, 12, np.std), color='#16a34a', label='rolling std(12)')
    a.set_title(name, fontsize=10); a.legend(fontsize=7)
plt.tight_layout(); plt.show()
print('A drifting rolling mean is non-stationarity you can see without a test.')

**Reading the next cell.** The two tests point in opposite directions, and that
is the whole reason to run both:

| | null hypothesis | small p-value means |
|---|---|---|
| **ADF** | there *is* a unit root (non-stationary) | **stationary** |
| **KPSS** | the series *is* stationary | **non-stationary** |

So you want ADF p **below** 0.05 and KPSS p **above** it. If they disagree you
have a borderline series — usually trend-stationary rather than
difference-stationary — and the fix is detrending, not another difference.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings; warnings.filterwarnings('ignore')

def report(v, name):
    adf = adfuller(v, autolag='AIC')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        kp = kpss(v, regression='c', nlags='auto')
    print(f'{name:<22} ADF stat {adf[0]:>7.3f}  p {adf[1]:.4f} -> '
          f'{"stationary" if adf[1] < 0.05 else "NON-stationary":<15}'
          f'| KPSS p {kp[1]:.3f} -> {"stationary" if kp[1] > 0.05 else "NON-stationary"}')

report(y, 'original')
report(np.diff(y), 'diff(1)')
report(y[PERIOD:] - y[:-PERIOD], f'seasonal diff({PERIOD})')
report(np.diff(y[PERIOD:] - y[:-PERIOD]), f'diff(1) + diff({PERIOD})')

print('\nADF H0 = has a unit root (non-stationary): small p means STATIONARY.')
print('KPSS H0 = is stationary:                    small p means NON-stationary.')
print('They are opposite tests, so run BOTH — agreement is what you want.')

In [ ]:
# Differencing removes trend; it does not remove seasonality. That needs lag-m.
d1 = np.diff(y)
sd = y[PERIOD:] - y[:-PERIOD]
fig, ax = plt.subplots(1, 3)
for a, (v, name) in zip(ax, [(y, 'y'), (d1, 'diff(1) — trend gone, season remains'),
                             (sd, f'diff({PERIOD}) — season gone')]):
    a.plot(v, color=TRACK, lw=1.2); a.axhline(np.mean(v), color='grey', ls='--', lw=0.8)
    a.set_title(name, fontsize=9)
plt.tight_layout(); plt.show()

print('Over-differencing is a real failure mode: it inflates variance.')
for k in range(4):
    v = y.copy()
    for _ in range(k): v = np.diff(v)
    print(f'  d={k}: variance {v.var():>9.3f}')
print('Take the smallest d that passes the tests, not the largest.')

---
## ts-m5 · AR models, ACF/PACF, and intervals

ACF cuts off after `q` for an MA(q); PACF cuts off after `p` for an AR(p).
That pair of facts is the whole identification step.

**Reading the next cell.** ACF is easy: correlate the series with a shifted copy
of itself. PACF is the interesting one.

At lag 3, the plain ACF is large partly *because* lag 1 was large and lag 1
influences lag 2 influences lag 3 — the correlation is inherited, not direct.
PACF strips that out: `phi_kk` is the last coefficient of the best AR(k) fit,
i.e. what lag k adds **once lags 1..k-1 are already in the model**.

`pacf` below is the Levinson–Durbin recursion, which gets every `phi_kk`
without solving k separate least-squares problems: each step reuses the
previous step's coefficients (`phi[j] - pkk*phi[k-2-j]`) instead of refitting.

In [ ]:
def acf(y, max_lag):
    m = y.mean(); d = ((y-m)**2).sum()
    return np.array([1.0 if k == 0 else ((y[k:]-m)*(y[:-k]-m)).sum()/d
                     for k in range(max_lag+1)])

def pacf(y, max_lag):
    """Levinson-Durbin on the sample ACF: phi_kk is lag k's DIRECT contribution
    once lags 1..k-1 are already accounted for."""
    r = acf(y, max_lag); out = [1.0]; phi = []
    for k in range(1, max_lag+1):
        num, den = r[k], 1.0
        for j in range(k-1):
            num -= phi[j]*r[k-1-j]
            den -= phi[j]*r[j+1]
        pkk = 0.0 if abs(den) < 1e-12 else num/den
        phi = [phi[j] - pkk*phi[k-2-j] for j in range(k-1)] + [pkk]
        out.append(pkk)
    return np.array(out)

a, p = acf(y, 5), pacf(y, 5)
print('ACF  1..5:', a[1:].round(6))
print('PACF 1..5:', p[1:].round(6))
# The site's widgets show exactly these numbers.
assert np.allclose(a[1:], [0.879101, 0.717669, 0.551181, 0.391411, 0.274603], atol=1e-6)
assert np.allclose(p[1:], [0.879101, -0.242753, -0.091075, -0.074045, 0.078994], atol=1e-6)

In [ ]:
max_lag = 24
A, P = acf(y, max_lag), pacf(y, max_lag)
band = 1.96/np.sqrt(len(y))          # 95% "no autocorrelation" band

fig, ax = plt.subplots(1, 2)
for a_, v, name in [(ax[0], A, 'ACF'), (ax[1], P, 'PACF')]:
    a_.bar(range(len(v)), v, color=TRACK, width=0.4)
    a_.axhspan(-band, band, color='grey', alpha=0.2)
    a_.axhline(0, color='black', lw=0.8); a_.set_title(f'{name} (band = +/-1.96/sqrt(n))', fontsize=10)
plt.tight_layout(); plt.show()

print(f'significance band = +/-{band:.3f}')
print(f'ACF decays slowly -> trend/non-stationarity still present')
print(f'PACF first drops inside the band at lag {int(np.argmax(np.abs(P[1:]) < band)) + 1} -> suggests that p')

**Reading the next cell.** Yule–Walker turns fitting an AR(p) into solving a
`p x p` linear system rather than running an optimiser.

Multiply the AR equation by `y_{t-k}` and take expectations, and every term
becomes an autocorrelation: `rho_k = phi_1*rho_{k-1} + ... + phi_p*rho_{k-p}`.
Write that for k = 1..p and you get `R phi = r`, where `R` is the Toeplitz
matrix of sample autocorrelations (`R[i,j] = rho_|i-j|`) and `r` is
`[rho_1..rho_p]`. One `np.linalg.solve` and you are done.

In [ ]:
def fit_ar_yule_walker(y, p):
    """Solve the Yule-Walker system R phi = r on the sample ACF."""
    m = y.mean(); x = y - m
    r = acf(x, p)
    R = np.array([[r[abs(i-j)] for j in range(p)] for i in range(p)])
    phi = np.linalg.solve(R, r[1:p+1])
    fitted = np.full(len(y), np.nan)
    for i in range(p, len(y)):
        fitted[i] = sum(phi[k]*x[i-1-k] for k in range(p)) + m
    resid = (y - fitted)[~np.isnan(fitted)]
    return phi, m, (resid**2).mean()

phi, mu, sigma2 = fit_ar_yule_walker(y, 2)
print(f'AR(2) Yule-Walker  phi = {phi.round(6)}  mean {mu:.6f}  sigma^2 {sigma2:.6f}')
assert np.allclose(phi, [1.092505, -0.242753], atol=1e-6)      # matches the site

from statsmodels.tsa.ar_model import AutoReg
sm_fit = AutoReg(y, lags=2, old_names=False).fit()
print(f'AutoReg (OLS)      phi = {sm_fit.params[1:].round(6)}  const {sm_fit.params[0]:.4f}')
print('\nThey differ slightly on purpose: Yule-Walker solves the moment equations,')
print('AutoReg runs conditional-OLS. Both are consistent; neither is "the" answer.')

**Reading the next cell.** Forecasting `h` steps means feeding the model its own
predictions — that is the loop in `forecast_ar`. The interesting part is the
interval.

Any stationary AR can be rewritten as an infinite weighted sum of past shocks
(the MA(∞) form), and `psi[j]` is the weight on the shock `j` steps back. A
forecast `h` steps ahead misses `h` shocks that have not happened yet, so its
error variance is `sigma^2 * (psi[0]^2 + ... + psi[h-1]^2)` — a **cumulative
sum**, which is exactly why the band widens and never narrows.

In [ ]:
def psi_weights(phi, h):
    """MA(inf) weights — how a shock at time t still echoes h steps later."""
    psi = [1.0]
    for j in range(1, h):
        psi.append(sum(phi[k]*(psi[j-1-k] if j-1-k >= 0 else 0) for k in range(len(phi))))
    return np.array(psi)

def forecast_ar(y, phi, mu, sigma2, h, z=1.96):
    hist = list(y[-len(phi):] - mu); pt = []
    for _ in range(h):
        nxt = sum(phi[k]*hist[-1-k] for k in range(len(phi)))
        hist.append(nxt); pt.append(nxt + mu)
    psi = psi_weights(phi, h)
    half = z*np.sqrt(sigma2*np.cumsum(psi**2))
    return np.array(pt), half

pt, half = forecast_ar(y, phi, mu, sigma2, 12)
print(f'{"h":>3}{"forecast":>10}{"+/- 95%":>10}')
for i in (0, 1, 2, 5, 11):
    print(f'{i+1:>3}{pt[i]:>10.2f}{half[i]:>10.2f}')
print('\nThe interval widens with horizon because forecast-error variance is')
print('sigma^2 * cumsum(psi^2) — each extra step adds another unforecastable shock.')
assert np.all(np.diff(half) > 0), 'prediction intervals must widen monotonically'

fig, ax = plt.subplots()
ax.plot(range(len(y)), y, color=TRACK, label='observed')
fx = range(len(y), len(y)+12)
ax.plot(fx, pt, color='#dc2626', label='AR(2) forecast')
ax.fill_between(fx, pt-half, pt+half, color='#dc2626', alpha=0.18, label='95% interval')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## ts-m6 · ARMA, ARIMA, SARIMA

`ARIMA(p,d,q)` folds the differencing into the model so the forecast comes back
on the original scale automatically. `SARIMA(p,d,q)(P,D,Q)m` adds the same three
terms again at the seasonal lag.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings; warnings.filterwarnings('ignore')

for order in [(1,0,0), (0,0,1), (1,0,1), (1,1,1), (2,1,1)]:
    f = ARIMA(y, order=order).fit()
    print(f'ARIMA{order}  AIC {f.aic:>8.2f}  BIC {f.bic:>8.2f}')
print('\nLower AIC wins — but only compare models fitted to the SAME differencing,')
print('because differencing changes the number of observations the likelihood uses.')

**Reading the next cell — the notation.** `SARIMA(p,d,q)(P,D,Q)m` is the same
three terms applied twice, once at lag 1 and once at lag `m`:

- `p` / `P` — **AR**: how many past values feed the prediction (at lag 1 / lag m)
- `d` / `D` — **I**: how many times to difference (ordinary / seasonal)
- `q` / `Q` — **MA**: how many past *errors* feed the prediction
- `m` — the season length, 12 for monthly data

So `(1,1,1)(1,1,1)12` means: difference once, difference again at lag 12, then
fit one AR and one MA term at each of the two lags. In the summary table below
`ar.L1` is the ordinary AR term and `ar.S.L12` the seasonal one.

In [ ]:
# The seasonal model. (1,1,1)(1,1,1)_12 is the standard first thing to try.
fit = SARIMAX(s, order=(1,1,1), seasonal_order=(1,1,1,PERIOD),
              enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
print(fit.summary().tables[1])
print(f'\nAIC {fit.aic:.2f}  BIC {fit.bic:.2f}')

# Step 3 of Box-Jenkins: DIAGNOSE before you trust anything.
lb = acorr_ljungbox(fit.resid, lags=[6, 12], return_df=True)
print('\nLjung-Box (H0: residuals are white noise — want p > 0.05):')
print(lb.round(4))

In [ ]:
fit.plot_diagnostics(figsize=(10, 6)); plt.tight_layout(); plt.show()

fc = fit.get_forecast(steps=24)
mean_fc, ci = fc.predicted_mean, fc.conf_int(alpha=0.05)
fig, ax = plt.subplots()
ax.plot(s.index, s.values, color=TRACK, label='observed')
ax.plot(mean_fc.index, mean_fc.values, color='#dc2626', label='SARIMA forecast')
ax.fill_between(mean_fc.index, ci.iloc[:, 0], ci.iloc[:, 1], color='#dc2626', alpha=0.18)
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

width = (ci.iloc[:, 1] - ci.iloc[:, 0]).values
print(f'interval width at h=1: {width[0]:.2f}   at h=24: {width[-1]:.2f}   '
      f'({width[-1]/width[0]:.1f}x wider)')
assert width[-1] > width[0]

In [ ]:
# The Box-Jenkins loop, automated crudely. This is what pmdarima's auto_arima does.
import itertools, warnings; warnings.filterwarnings('ignore')

best, rows = None, []
for p, d, q, P, D, Q in itertools.product([0,1,2],[1],[0,1,2],[0,1],[1],[0,1]):
    try:
        f = SARIMAX(s, order=(p,d,q), seasonal_order=(P,D,Q,PERIOD),
                    enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
        rows.append(((p,d,q),(P,D,Q,PERIOD), f.aic))
        if best is None or f.aic < best[2]: best = ((p,d,q),(P,D,Q,PERIOD), f.aic)
    except Exception:
        pass

rows.sort(key=lambda r: r[2])
print(f'{"order":<12}{"seasonal":<16}{"AIC":>9}')
for o, so, aic in rows[:8]:
    print(f'{str(o):<12}{str(so):<16}{aic:>9.2f}')
print(f'\nbest by AIC: SARIMA{best[0]}{best[1]}  AIC {best[2]:.2f}')
print('\nDo not stop at the AIC. Refit the winner, run Ljung-Box on ITS residuals,')
print('and check the forecast against a held-out window — AIC is in-sample fit')
print('with a complexity penalty, not evidence that the model forecasts well.')

In [ ]:
# The honest evaluation: refit on train only, score on the held-out tail.
tr, te = s.iloc[:split], s.iloc[split:]
res = {}
for name, mk in [
    ('seasonal naive', lambda: np.array([tr.values[-PERIOD + (i % PERIOD)] for i in range(len(te))])),
    ('Holt-Winters',   lambda: ExponentialSmoothing(tr, trend='add', seasonal='add',
                                seasonal_periods=PERIOD, initialization_method='estimated'
                              ).fit().forecast(len(te)).values),
    ('SARIMA(1,1,1)(1,1,1)', lambda: SARIMAX(tr, order=(1,1,1), seasonal_order=(1,1,1,PERIOD)
                              ).fit(disp=False).forecast(len(te)).values),
]:
    try:
        pred = mk()
        res[name] = np.sqrt(((te.values - pred)**2).mean())
    except Exception as e:
        res[name] = float('nan')

print(f'{"model":<26}{"test RMSE":>10}')
for k, v in sorted(res.items(), key=lambda kv: kv[1]):
    print(f'{k:<26}{v:>10.3f}')
print('\nIf the fancy model does not beat seasonal naive here, ship seasonal naive.')

---
## Where to go next

- **Multiple series at once** — `statsmodels.tsa.statespace.VARMAX`, or just fit
  one model per series and compare against a global baseline.
- **Exogenous drivers** — `SARIMAX(..., exog=X)` for promotions, holidays, price.
- **Prophet / NeuralProphet** — trend changepoints plus holiday effects, robust
  to missing data; worth trying when the seasonality is irregular.
- **Backtesting properly** — expanding-window (`TimeSeriesSplit`) rather than one
  held-out tail, so the score is not a single lucky window.

Change `seed=7` in `generate_series` and re-run the whole notebook: every
conclusion above should survive, and the ones that do not were noise.